# 01 — Data Exploration
## AI City Twin Casablanca

Explore the city's road network, tram lines, POIs, and population grid.

In [ ]:
import sys
sys.path.insert(0, '..')

import geopandas as gpd
import matplotlib.pyplot as plt
import folium
from src.data_prep import (
    fetch_roads_offline, generate_synthetic_tram_lines,
    generate_synthetic_pois, generate_population_grid,
)
from src.utils import CASABLANCA_CENTER

In [ ]:
# Load datasets
roads = fetch_roads_offline()
tram = generate_synthetic_tram_lines()
pois = generate_synthetic_pois()
pop_grid = generate_population_grid()

print(f'Roads: {len(roads)}')
print(f'Tram lines: {len(tram)}')
print(f'POIs: {len(pois)}')
print(f'Pop grid cells: {len(pop_grid)}')

In [ ]:
# Plot roads by type
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
roads.plot(ax=ax, column='road_type', legend=True, linewidth=0.5)
tram.plot(ax=ax, color='red', linewidth=3, label='Tram')
ax.set_title('Casablanca Road Network')
plt.tight_layout()
plt.show()

In [ ]:
# POI distribution
print(pois['poi_type'].value_counts())

fig, ax = plt.subplots(1, 1, figsize=(12, 10))
pop_grid.plot(ax=ax, column='population', cmap='YlOrRd', legend=True, alpha=0.6)
pois.plot(ax=ax, column='poi_type', markersize=10, legend=True, alpha=0.7)
ax.set_title('Population Density & POIs')
plt.tight_layout()
plt.show()

In [ ]:
# Interactive map
m = folium.Map(location=[CASABLANCA_CENTER[0], CASABLANCA_CENTER[1]], zoom_start=12)

# Add tram lines
for _, row in tram.iterrows():
    coords = [(c[1], c[0]) for c in row.geometry.coords]
    folium.PolyLine(coords, color='blue', weight=4).add_to(m)

# Add sample POIs
for _, row in pois.head(100).iterrows():
    folium.CircleMarker(
        [row.geometry.y, row.geometry.x], radius=3,
        color='orange', fill=True,
        popup=f"{row['name']} ({row['poi_type']})"
    ).add_to(m)

m